In [1]:
#pip install torch torchvision opencv-python

#TODO
#Split training data into 90-10 train, val(small validation set should speed up training)
#Implement Freezing early weights
#Do proper eval

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import v2
from os.path import isfile

from datetime import datetime           #File timstamp
import time                             #T+121239
from PIL import Image                   #Needed to load random image
import cv2 as cv                        #Show images and stuff(Can also be done with PIL
import matplotlib.pyplot as plt         #Show images for colab
import numpy as np                      #Honestly this should just be a core library

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
Timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

In [3]:
class LeNet(nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.fc_1 = nn.Linear(16 * 4 * 4, 120)
        self.fc_2 = nn.Linear(120, 84)
        self.fc_3 = nn.Linear(84, output_dim)

    def forward(self, x):
        x = self.conv1(x)
        x = F.max_pool2d(x, kernel_size=2)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.max_pool2d(x, kernel_size=2)
        x = F.relu(x)
        x = x.view(x.shape[0], -1)
        x = self.fc_1(x)
        x = F.relu(x)
        x = self.fc_2(x)
        x = F.relu(x)
        x = self.fc_3(x)
        return x

In [4]:
#Pretrained model weight None>Random, "DEFAULT">torch MobileNetV2, "IMAGENET1K_V1">original MobileNetV2, or "FILENAME" for custom
PTWeights = "DEFAULT"
PTFreeze = False                             #Freeze Non-Output layer
PTOutputs = 7                             #Alter Output size if different from loaded network (WILL wipe weights for this)
TrainingRate = 0.01
TrainingEpochs = 0
EarlyStop = 10

model = LeNet(PTOutputs)
if PTWeights and isfile(PTWeights): model.load_state_dict(torch.load(PTWeights, weights_only=True))

PreProcess = v2.Compose([                   #Preprocessing steps to get dataset to appropriate format
    #v2.Resize(224),                         #Resizes to 224px short side
    #v2.Resize(None, max_size=224),          #Resizes to 224px long side
    #v2.Pad(40),
    v2.Grayscale(1),
    v2.CenterCrop(28),                     #Crops excess on long side
    v2.ToImage(),                           #Makes it an image???
    v2.ToDtype(torch.float32, scale = True) #Changes datatype to float and value range from 0-255 to 0-1
    #v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),    #Normalizes image because apparently the (pretrained)model likes that
])

#if PTOutputs != model.classifier[1].out_features: model.classifier[1] = nn.Linear(model.classifier[1].in_features, PTOutputs)
if PTFreeze:
    for param in model.parameters():
        param.requires_grad = False
    model.classifier[-1].weight.requires_grad = True
    model.classifier[-1].bias.requires_grad = True

In [5]:
AnnotationLevel = "family"
AnnotationFile = "./fgvc-aircraft-2013b/data/families.txt"

In [ ]:
txt = np.genfromtxt('train/gt.txt',delimiter =';', dtype= None,encoding=None)

#Creating a dictionary with image names as key and annotations as value
dic ={}
for i in range (0,len(txt)):
    #Image name is first element of annotation file
    img_name = txt[i][0]
    # 4 Coordinates
    target = [txt[i][1],txt[i][2],txt[i][3],txt[i][4],txt[i][5]]
    #Last element is the class number
    clas = txt[i][-1]
    #If multiple objects, store coordinates and classes as list of lists
    if(img_name in dic):
        dic[img_name].append(target)
    else:
        dic[img_name] = [target]
print(dic['00001.ppm'])
print("Number of Images: " + str(len(dic)))

In [ ]:
#Copy only files that are annotated in the gt.txt to imagesf
import shutil
import glob
pt = glob('./train/*.ppm')

len(pt)
#Copying into new directory
for i in range(len(dic)):
    ofile = r'./train/{}'.format(list(dic)[i])
    target = r'./train/imagesf/{}'.format(list(dic)[i])
    shutil.copyfile(ofile, target)
#Check if len(dic) == number of images in folder
print(len(glob('./train/imagesf/*.ppm')))
len(dic)

In [ ]:
class myDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms
        # load all image files, sorting them to ensure that they are aligned
        self.imgs = list(sorted(os.listdir(os.path.join(root, "imagesf"))))
 
    def __getitem__(self, idx):
        # Load image path
        img_path = os.path.join(self.root, "imagesf", self.imgs[idx])
        #Load image as PIL
        img = Image.open(img_path).convert("RGB")        
        # Get objects in the image
        objects = dic[self.imgs[idx]]
        # Get bounding box coordinates for each object in image
        boxes = []
        labels = []
        for obj in objects:
            #print(idx, obj[-1], self.imgs)
            name = obj[-1]
            labels.append(np.int(name))
            #Get bounding box coordinates
            xmin = np.float(obj[0])
            ymin = np.float(obj[1])
            xmax = np.float(obj[2])
            ymax = np.float(obj[3])
            boxes.append([xmin, ymin, xmax, ymax])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)        
 
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        # suppose all instances are not crowd
        iscrowd = torch.zeros((len(objects),), dtype=torch.int64)
 
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = image_id
        target["area"] = area
        target["iscrowd"] = iscrowd
 
        if self.transforms is not None:
            # Note that target (including bbox) is also transformed\enhanced here, which is different from transforms from torchvision import
            # Https://github.com/pytorch/vision/tree/master/references/detectionOfTransforms.pyThere are examples of target transformations when RandomHorizontalFlip
            img, target = self.transforms(img, target)
 
        return img, target
 
    def __len__(self):
        return len(self.imgs)

In [ ]:
#TrainData = datasets.FGVCAircraft(root="./", split="train", annotation_level=AnnotationLevel, download=True, transform=PreProcess)
#TestData = datasets.FGVCAircraft(root="./", split="test", annotation_level=AnnotationLevel, download=True, transform=PreProcess)
root="../../Datasets/GTSDB/"
TrainData = myDataset(root, PreProcess)
TestData = myDataset(root, PreProcess)
TrainDataLoader = torch.utils.data.DataLoader(TrainData, batch_size=80, shuffle=True, pin_memory=True)#, num_workers=args.nThreads
TestDataLoader = torch.utils.data.DataLoader(TestData, batch_size=80, shuffle=True, pin_memory=True)#, num_workers=args.nThreads

In [9]:
annotations = ["You are at peace", "Right", "Left", "RotRight", "RotLeft", "SoftDrop", "HardDrop"]
#with open(AnnotationFile, 'rb') as file:
#    for line in file:
#        annotations.append(line.decode("utf-8").strip())

In [10]:
criterion = nn.CrossEntropyLoss()       #Multiclass classification
optimizer = torch.optim.Adam(model.parameters(), lr=TrainingRate)
def TrainEpoch():                       #Trains once on all training data
    accumulatedLoss = 0.0
    #Train on the whole dataset but tragically my GPU doesnt fit the whole dataset
    #(which is weird because its only a few GB) so it has to be in batches
    for BatchID, Batch in enumerate(TrainDataLoader):
        BatchIm, BatchLabel = Batch
        BatchIm, BatchLabel = BatchIm.to(device), BatchLabel.to(device)     #Put stuff on GPU(if cuda is available)

        optimizer.zero_grad()                   #Reset gradients
        output = model(BatchIm)                 #Make predictions
        loss = criterion(output, BatchLabel)    #How wrong are we
        loss.backward()                         #Change model weights based on how wrong we are
        optimizer.step()
        accumulatedLoss += loss.item()
    return accumulatedLoss / TrainDataLoader.batch_size

In [11]:
def TrainEpochs(Epochs):
    model.to(device)
    BestTest = np.inf
    NoImprovement = 0
    tStart = time.time()
    for Epoch in range(Epochs):
        model.train(True)
        print(f"Train T+{time.time() - tStart:>5.0f}")
        LossTrain = TrainEpoch()

        LossTest = 0.0
        model.eval()
        with torch.no_grad():
            print(f"Eval  T+{time.time() - tStart:>5.0f}")
            for BatchID, Batch in enumerate(TestDataLoader):
                BatchIm, BatchLabel = Batch
                BatchIm, BatchLabel = BatchIm.to(device), BatchLabel.to(device)     #Put stuff on GPU(if cuda is available)
                output = model(BatchIm)                                             #Make predictions
                LossTest += criterion(output, BatchLabel)                           #How wrong are we

            LossTest = LossTest / TestDataLoader.batch_size

        if LossTest < BestTest:
            BestTest, NoImprovement = LossTest, 0
            torch.save(model.state_dict(), f"Model_{Timestamp}_{Epoch+1}")
        else:
            NoImprovement += 1
            if NoImprovement > EarlyStop:
                print(f"Training does not continue until morale improves")
                break
        print(f"Epoch [{Epoch+1:>3}/{Epochs:>3}] T+{time.time() - tStart:>5.0f}, Training Loss: {LossTrain:>6.3f}, Test Loss: {LossTest:>6.3f},  Worse then best [{NoImprovement:>2}/{EarlyStop:>2}]")

In [12]:
TrainEpochs(TrainingEpochs)

In [13]:
numberID = 207
testImage, Label = TestData[numberID]
print(f"It is a ({Label}){annotations[Label]}", testImage.shape)
testTensor = PreProcess(testImage).unsqueeze(0).to(device)  #Put the image through the preprocessing, then unsqueeze to add an extra dimension(image is 3D this model wants 4D

result = model(testTensor)
probabilities = F.softmax(result, dim=1).squeeze()
topProb, topClass = torch.topk(probabilities, 5)
for i in range(len(topProb)):
    print(f"I am {topProb[i].item():.2f} percent sure this is a {"annotations[topClass[i].item()]"} Class:{topClass[i].item()}")

It is a (5)SoftDrop torch.Size([1, 28, 28])
I am 0.15 percent sure this is a annotations[topClass[i].item()] Class:2
I am 0.15 percent sure this is a annotations[topClass[i].item()] Class:1
I am 0.15 percent sure this is a annotations[topClass[i].item()] Class:6
I am 0.15 percent sure this is a annotations[topClass[i].item()] Class:4
I am 0.14 percent sure this is a annotations[topClass[i].item()] Class:3


In [14]:
#from google.colab.patches import cv2_imshow
#testImage, Label = TestData[numberID]
#testImage = testImage.permute(1, 2, 0).numpy() * 200
#cv2_imshow(testImage)
#print(Label)

In [15]:
def count_parameters(model):
    print(f"{"Layer name":^30}|{"Parameters":>10}|{"Trainable":>10}")
    total_params, train_params = 0, 0
    for name, parameter in model.named_parameters():
        #if not parameter.requires_grad: continue
        params = parameter.numel()
        print(f"{name:<30}|{params:>10}|{params*parameter.requires_grad:>10}")
        total_params += params
        if parameter.requires_grad: train_params += params
    print(f"{"Total parameters":<30}|{total_params:>10}|{train_params:>10}")
count_parameters(model)

          Layer name          |Parameters| Trainable
conv1.weight                  |       150|       150
conv1.bias                    |         6|         6
conv2.weight                  |      2400|      2400
conv2.bias                    |        16|        16
fc_1.weight                   |     30720|     30720
fc_1.bias                     |       120|       120
fc_2.weight                   |     10080|     10080
fc_2.bias                     |        84|        84
fc_3.weight                   |       588|       588
fc_3.bias                     |         7|         7
Total parameters              |     44171|     44171


Some things i learned:
freezing all parameters except classifier layer reduces GPU usage during training a *lot*, so we might be able to train a model on UCloud decently fast if we just fine tune? It does not actually train faster on my computer but thats because CPU is the limit for me

possibly because of backpropagation not needing to calculate so many gradients, possibly due to just not tracking gradients
fully trimmed model i tested also replaced classifier layer with one that only had 100 outputs instead of 1000
since MobileNetV2 is trained on a dataset with 1000 classes and we only have a bit under 100

this means it only trains 128100 parameters instead of 3504872, and is what is called "fine tuning"(i think) since you leave almost the whole model intact and just retrain the output layers

the fully trimmed down model used only about 2GB VRAM and barely any work on GPU

In [16]:
print(model)

LeNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc_1): Linear(in_features=256, out_features=120, bias=True)
  (fc_2): Linear(in_features=120, out_features=84, bias=True)
  (fc_3): Linear(in_features=84, out_features=7, bias=True)
)
